In [ ]:
!pip install "pymilvus[model]"
!pip install langchain-milvus peft

In [3]:
## following Milvus documentation https://milvus.io/docs/insert-update-delete.md
from pymilvus import MilvusClient
from qwen3embedding import Qwen3EmbeddingModel
import numpy as np


## read from saved collection 
client = MilvusClient(
    uri="http://localhost:19530"
    )

embedding=Qwen3EmbeddingModel()

#client.describe_collection(COLLECTION_NAME)
def search_collection(query, topk=2, client=client, embedding=embedding):    
    COLLECTION_NAME = "SovAISwedish"
    EMBEDDING_DIM=1024
    query_embeddings = embedding.get_embeddings(query, prompt_name="query")
    query_embeddings = np.float32(query_embeddings)        
    OUTPUT_FIELDS = ['id', 'chunk', 'source']    
    search_params = {"metric_type": "IP","params":{"nprobe":1024}}
    TOP_K = topk    
    results = client.search(
        COLLECTION_NAME,
        data=[query_embeddings],
        limit=TOP_K,
        search_params=search_params,
        output_fields=OUTPUT_FIELDS,
        consistency_level="Eventually")
    return results




In [4]:
query="förklarar svenskautbildningssystemet för mig, tack"
results=search_collection(query)


embed a query


In [13]:
results

data: [[{'id': 0, 'distance': 0.631366491317749, 'entity': {'id': 0, 'chunk': 'Det svenska utbildningssystemet är uppbyggt i flera steg och inkluderar obligatoriska och  \nfrivilliga utbildningar. Den grundläggande utbildningen omfattar förskoleklass och grundskola,  \nsom är obligatoriska för alla barn och ungdomar. Efter grundskolan finns det ett valfritt  \ngymnasium, följt av högre studier på universitet, högskola eller yrkeshögskola.  \nUtbildningssystemet i korthet:  \n- Förskola: Rätt att gå för barn mellan 1 och 5 år.  \n- Förskoleklass: Obligatorisk från det år barnet fyller sex år.  \n- Grundskola: Obligatorisk och omfattar 9 årskurser.  \n- Gymnasium: Frivillig utbildning efter grundskolan, som bereder för högre studier.  \n- Högskola/Universitet/Yrkeshögskola: Högre utbildning för vuxna.  \n- Sfi (Svenska för invandrare): Utbildning för invandrare som vill lära sig svenska.  \n- Komvux (Kommunal vuxenutbildning): Utbildning för vuxna på grundläggande och  \n- gymnasial nivå

In [12]:
results[0][0]["entity"]["chunk"]

'Det svenska utbildningssystemet är uppbyggt i flera steg och inkluderar obligatoriska och  \nfrivilliga utbildningar. Den grundläggande utbildningen omfattar förskoleklass och grundskola,  \nsom är obligatoriska för alla barn och ungdomar. Efter grundskolan finns det ett valfritt  \ngymnasium, följt av högre studier på universitet, högskola eller yrkeshögskola.  \nUtbildningssystemet i korthet:  \n- Förskola: Rätt att gå för barn mellan 1 och 5 år.  \n- Förskoleklass: Obligatorisk från det år barnet fyller sex år.  \n- Grundskola: Obligatorisk och omfattar 9 årskurser.  \n- Gymnasium: Frivillig utbildning efter grundskolan, som bereder för högre studier.  \n- Högskola/Universitet/Yrkeshögskola: Högre utbildning för vuxna.  \n- Sfi (Svenska för invandrare): Utbildning för invandrare som vill lära sig svenska.  \n- Komvux (Kommunal vuxenutbildning): Utbildning för vuxna på grundläggande och  \n- gymnasial nivå.'

In [6]:
import getpass
import os

# del os.environ['NVIDIA_API_KEY']  ## delete key and reset
if os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    print("Valid NVIDIA_API_KEY already in environment. Delete to reset")
else:
    nvapi_key = getpass.getpass("NVAPI Key (starts with nvapi-): ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
global nvapi_key

NVAPI Key (starts with nvapi-):  ········


In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_nvidia_ai_endpoints import ChatNVIDIA

client = ChatNVIDIA(
  model="qwen/qwen3-235b-a22b",
  api_key=os.environ["NVIDIA_API_KEY"], 
  temperature=0.2,
  top_p=0.7,
  max_tokens=8192,
  extra_body={"chat_template_kwargs": {"thinking":True}},
)


C:\Users\zcharpy\AppData\Local\anaconda3\envs\py312\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:212: UserWarning: Found qwen/qwen3-235b-a22b in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [ ]:
import json
import re
import os

import yaml
from colorama import Fore
from dotenv import load_dotenv
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import requests
import json
from typing import Any, Dict, Iterator, List, Optional
from langchain_core.callbacks import (
    CallbackManagerForLLMRun,
)
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import (
    AIMessage,
    AIMessageChunk,
    BaseMessage,
)
from langchain_core.messages.ai import UsageMetadata
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult
from pydantic import Field
from langchain_core.prompts import PromptTemplate


class NemotronChatModel(BaseChatModel):
    """
    Hacky implementation of the Nemotron 49b endpoints into langchain BaseChatModel
    """  
    
    alt_tools: dict = {
    "get_jira_issues_tool":
        "Queries internal project management systems like Jira to retrieve tickets related to a specific customer issues, feature "
        "or task. Outputs a JSON file containing ticket descriptions, statuses, and priorities."
        "A tool that fetches information about all the issues from a given Jira project. "
        "This information can be used to check if there are existing works or tickets by internal team related "
        "to a keyword. The tool can also be used to determine the developer with the best skills to complete "
        "a given issue. If an issue has a status of 'Done', it has been completed and is closed. "
        "For example, if a developer has been assigned to many issues pertaining to API development, "
        "they are likely a good candidate for new issues pertaining to API development and/or back end development."
        "To get all jira issues, this tool takes an empty string as input. "
        "To get one jira issue, this tool takes the issue key, which will start with 'GTC-' and "
        "end with a number. "
        "The following are valid examples of an issue key: 'GTC-2', 'GTC-543', 'GTC-1120'. "
        "Nothing should be passed to this tool besides either an empty string or an issue key as a string.",
    "forum_search_tool": "Searches public or internal forums for discussions related to specific topics or keywords. "
                         "Outputs a summarization and count.",
    "search_crm_tool":
        "Searches the CRM system for customer tickets or requests based on specific keywords or phrases. "
        "Outputs a JSON file containing relevant tickets and their details.",
    "summarizer_tool": "summarize",
    "final_response": "use this when you are ready to response to the user, otherwise please select other tool",
    }
        
    # if you think you are ready to response to the user, then use the tool 'final_response' and leave the tool_input empty.
    sys_prompt : str = """    
    You are an expert reasoning model tasked with creating a detailed step-by-step execution plan
    from user input : 
    <input>
    {input}
    </input>
    for a system that has the following description:
    You have access to the following tools and their descriptions:
    <tools>
    {tools}
    </tools>
    
    You should STRICTLY follow the below format when you construct a plan
    <format>
    - you should ALWAYS start with '**PLAN:**' in the first line to mark the begining of the Step-by-Step Execution Plan
    - you should number the individual step using 1., 2., 3. and so on
    - for each step, you should include call to a tool in JSON format, see below example of a tool call:
        {{"tool": "forum_search_tool", "tool_input": "Query for searching NVIDIA Develop Forums,
        e.g. 'Agentic profiling after:2024-08-18'"}}
    - the last 2 steps will ALWAYS be 'summarizer_tool', and 'final_response' such as below example plan demonostrate
    - return the plan only and nothing else
    An example plan could look like this:\n\n"
    1. {{tool: A , tool_input: X}}\n\n
    2. {{tool: B , tool_input: Y}}\n\n
    3. {{tool: C , tool_input: Z}}\n\n
    4. {{tool: summarizer_tool: output from previous 3 steps}}\n\n
    5. {{tool: final_response , tool_input: output from previous summarizer_tool step }}\n\n
    </format>
    **PLAN:**\n
    """
    sys_prompt_template: PromptTemplate=PromptTemplate(
        input_variables=["input", "tools"],
        template=sys_prompt,)

    model_name: str = 'stg/nvidia/llama-3.3-nemotron-49b-instruct-v1'
    def response(self, query):
        api_key=os.environ["NeMoTron_API"]
        headers = {
            'Content-Type': 'application/json',
            'Authorization': f"Bearer {api_key}",
        }
        
        json_data = {
            'model': model_name,
            'messages': [
                {
                    'role': 'user',
                    'content': query,
                },
            ],
            'temperature': 0.01,
            'top_p': 1,
            'max_tokens': 4096,
            'stream': False,
        }
        
        response = requests.post('https://integrate.api.nvidia.com/v1/chat/completions', headers=headers, json=json_data)

        if response.status_code==200:
            output=response.json()
            final_output=output["choices"][0]["message"]["content"]
        else:
            final_output="endpoint does not work this time, please contact admin or try again"
            
        return final_output
    async def _generate(
    self,
    query:str,
    **kwargs: Any,
    ) -> ChatResult:
        """Override the _generate method to implement the chat model logic.
    
        This can be a call to an API, a call to a local model, or any other
        implementation that generates a response to the input prompt.
    
        Args:
            messages: the prompt composed of a list of messages.
            stop: a list of strings on which the model should stop generating.
                  If generation stops due to a stop token, the stop token itself
                  SHOULD BE INCLUDED as part of the output. This is not enforced
                  across models right now, but it's a good practice to follow since
                  it makes it much easier to parse the output of the model
                  downstream and understand why generation stopped.
            run_manager: A run manager with callbacks for the LLM.
        """
         # Insert custom processing logic to generate a prompt from the messages
        
        prompt_strig=self.sys_prompt_template.template.format(input=query, tools='\n'.join([json.dumps(tool) for tool in self.alt_tools]))
        response =  self.response(prompt_strig) # <-- Follow the API guidelines for the model provider
    
        return response

    @property
    def _llm_type(self) -> str:
        """Get the type of language model used by this chat model."""
        return "nemotron_49b"

    @property
    def _identifying_params(self) -> Dict[str, Any]:
        """Return a dictionary of identifying parameters.

        This information is used by the LangChain callback system, which
        is used for tracing purposes make it possible to monitor LLMs.
        """
        return {
            # The model name allows users to specify custom token counting
            # rules in LLM monitoring applications (e.g., in LangSmith users
            # can provide per token pricing for their model and monitor
            # costs for the given LLM.)
            "model_name": self.model_name,
        }

nemotron_super=NemotronChatModel()

